# Datasets Exploration

Esplora i CSV prodotti da `scripts/data_summary.py` in `reports/datasets/` — uno per dataset, colonne `subject` + una colonna `present`/`missing` per ogni voce del registro.

## Report csv

In [1]:
from pathlib import Path

import pandas as pd

REPORTS_DIR = Path("/home/etosato/Projects/NEMESIS_fs/reports/datasets")
CSV_FILES = sorted(REPORTS_DIR.glob("data_summary__*.csv"))
CSV_FILES

[PosixPath('/home/etosato/Projects/NEMESIS_fs/reports/datasets/data_summary__UKLFR_stroke_UKLFR.csv'),
 PosixPath('/home/etosato/Projects/NEMESIS_fs/reports/datasets/data_summary__UNIPD_PASPORT.csv'),
 PosixPath('/home/etosato/Projects/NEMESIS_fs/reports/datasets/data_summary__UNIPD_PSP.csv'),
 PosixPath('/home/etosato/Projects/NEMESIS_fs/reports/datasets/data_summary__UNIPD_WashU.csv')]

Carica ogni CSV in un dict `{nome_dataset: DataFrame}`, usando il nome file (senza prefisso/suffisso) come chiave.

In [2]:
datasets: dict[str, pd.DataFrame] = {}

for path in CSV_FILES:
    name = path.stem.removeprefix("data_summary__")
    datasets[name] = pd.read_csv(path)

list(datasets.keys())


['UKLFR_stroke_UKLFR', 'UNIPD_PASPORT', 'UNIPD_PSP', 'UNIPD_WashU']

### Soggetti totali per dataset

Numero di righe (soggetti) in ciascun CSV.

In [3]:
totals = {name: len(df) for name, df in datasets.items()}

for name, n in totals.items():
    print(f"{name}: {n} soggetti")


UKLFR_stroke_UKLFR: 735 soggetti
UNIPD_PASPORT: 97 soggetti
UNIPD_PSP: 237 soggetti
UNIPD_WashU: 251 soggetti


### Missing per colonna
Per ogni dataset, conta quante righe hanno `missing` in ciascuna colonna dati (tutte tranne `subject`), sia in valore assoluto che percentuale sul totale soggetti.

In [4]:
for name, df in datasets.items():
    data_columns = [c for c in df.columns if c != "subject"]
    n_subjects = len(df)

    print(f"=== {name} ({n_subjects} soggetti) ===")
    for col in data_columns:
        n_missing = (df[col] == "missing").sum()
        pct = 100 * n_missing / n_subjects
        print(f"  {col}: {n_missing} missing ({pct:.1f}%)")
    print()


=== UKLFR_stroke_UKLFR (735 soggetti) ===
  lesion/manual_masks/anat/lesion_mask: 38 missing (5.2%)
  lesion/raw/anat/lesion_roi: 0 missing (0.0%)

=== UNIPD_PASPORT (97 soggetti) ===
  lesion/manual_masks/anat/lesion_mask: 14 missing (14.4%)
  lesion/raw/anat/lesion_roi: 14 missing (14.4%)

=== UNIPD_PSP (237 soggetti) ===
  lesion/manual_masks/anat/lesion_mask: 69 missing (29.1%)
  lesion/raw/anat/lesion_roi: 237 missing (100.0%)

=== UNIPD_WashU (251 soggetti) ===
  lesion/manual_masks/anat/lesion_mask: 49 missing (19.5%)
  lesion/raw/anat/lesion_roi: 49 missing (19.5%)



### Mismatch per riga

Per ogni soggetto, confronta i valori di tutte le colonne dati: se coincidono (tutte `present` o tutte `missing`) la riga è coerente, altrimenti è un mismatch (presente in una colonna, mancante in un'altra). Stampa il conteggio e il dettaglio delle righe in mismatch per ciascun dataset.

In [5]:
for name, df in datasets.items():
    data_columns = [c for c in df.columns if c != "subject"]

    is_mismatch = df[data_columns].nunique(axis=1) > 1
    n_mismatch = is_mismatch.sum()

    print(f"=== {name} ===")
    print(f"  coincidenti: {len(df) - n_mismatch} / {len(df)}")
    print(f"  mismatch:    {n_mismatch} / {len(df)}")

    if n_mismatch > 0:
        display(df.loc[is_mismatch, ["subject", *data_columns]])
    print()


=== UKLFR_stroke_UKLFR ===
  coincidenti: 697 / 735
  mismatch:    38 / 735


,subject,lesion/manual_masks/anat/lesion_mask,lesion/raw/anat/lesion_roi
4,sub-STUKLFR0005,missing,present
29,sub-STUKLFR0030,missing,present
34,sub-STUKLFR0035,missing,present
55,sub-STUKLFR0056,missing,present
96,sub-STUKLFR0097,missing,present
130,sub-STUKLFR0131,missing,present
131,sub-STUKLFR0132,missing,present
141,sub-STUKLFR0142,missing,present
150,sub-STUKLFR0151,missing,present
154,sub-STUKLFR0155,missing,present



=== UNIPD_PASPORT ===
  coincidenti: 97 / 97
  mismatch:    0 / 97

=== UNIPD_PSP ===
  coincidenti: 69 / 237
  mismatch:    168 / 237


,subject,lesion/manual_masks/anat/lesion_mask,lesion/raw/anat/lesion_roi
0,sub-STUNIPD0252,present,missing
1,sub-STUNIPD0253,present,missing
2,sub-STUNIPD0254,present,missing
3,sub-STUNIPD0255,present,missing
4,sub-STUNIPD0256,present,missing
...,...,...,...
225,sub-STUNIPD0477,present,missing
231,sub-STUNIPD0483,present,missing
233,sub-STUNIPD0485,present,missing
234,sub-STUNIPD0486,present,missing



=== UNIPD_WashU ===
  coincidenti: 251 / 251
  mismatch:    0 / 251

